# Advanced Analytics for a Better World — Lecture 2

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/aabw/notebooks/lecture-2/elizabeth-facility-location.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/aabw/notebooks/lecture-2/elizabeth-facility-location.ipynb)

> Original Lecture 2 notebook, retained with its author attribution and content.


# Data Science Essentials: Applied Optimization

Joaquim Gromicho, 2021

This notebook is part of the module Applied Optimization of the Analytics Academy's Data Science Essentials.

---
 > During this course we make use of Jupyter notebooks hosted by [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb).
  Notebooks deployed on `colab` require neither python nor other dependencies to be installed on your own machine, you only need a browser (preferably `chrome`) and you may also need a google account if you want to execute them.

---

Let us suppose now that Caroline is so successful that she considers opening a number of distribution centers to support scaling up her production.

Taking into account her customers $I$ and a set $J$ of possible locations for her new distribution centers she estimates the costs $c_j$ of opening at $j \in J$ and $d_{ij}$ of serving $i \in I$ from $j \in J$.

Her decision variables are:
     
$$
   x_j = \left\{
     \begin{array}{lr}
       1 & \mbox{center } j \mbox{ is built}\\
       0 & \mbox{otherwise}\\
     \end{array}
   \right.
       \mbox{ and }
   y_{ij} = \left\{
     \begin{array}{lr}
       1 & \mbox{customer } i \mbox{ is served at } j\\
       0 & \mbox{otherwise}\\
     \end{array}
   \right.
$$

Minimize facility-opening and service costs while assigning each customer to an open facility. Let $n=|I|$ be the number of customers:
$$
\begin{array}{rrcll}
\min    & \sum_{j\in J} c_jx_j + \sum_{i \in I, j \in J} d_{ij}y_{ij}\\
s.t.    & \sum_{j\in J} y_{ij}     & =    & 1     & \forall i \in I \\
        & \sum_{i\in I} y_{ij}     & \leq & n x_j & \forall j \in J \\
        & y_{ij} \in \{0,1\}       &      &       & \forall i \in I, j \in J \\
        & x_j \in \{0,1\}          &      &       & \forall j \in J \\
\end{array}
$$
       
The disaggregated model which is claimed to be stronger is:
$$
\begin{array}{rrcll}
\min    & \sum_{j\in J} c_jx_j + \sum_{i \in I, j \in J} d_{ij}y_{ij}\\
s.t.    & \sum_{j\in J} y_{ij}     & =    & 1     & \forall i \in I \\
        &               y_{ij}     & \leq & x_j   & \forall i \in I, j \in J \\
        & y_{ij} \in \{0,1\}       &      &       & \forall i \in I, j \in J \\
        & x_j \in \{0,1\}          &      &       & \forall j \in J \\
\end{array}
$$


In [ ]:
%matplotlib inline


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'pyomo': 'pyomo', 'highspy': 'highspy', 'numpy': 'numpy', 'matplotlib': 'matplotlib', 'pandas': 'pandas'}
ensure_packages(required_packages)
from teaching_utils import install_coin_solvers, solve_checked, available_pyomo_solvers
install_coin_solvers()
import pyomo.environ as pyo
import numpy as np
import pandas as pd
from time import perf_counter
solver_name = 'appsi_highs'
comparison_rows = []


In [ ]:
def FacilityLocationWeak( installation, service, solver=solver_name ):
    started = perf_counter()
    from pyomo.environ import ConcreteModel, Var, Objective, Constraint, NonNegativeReals, Binary, minimize

    model = ConcreteModel("Facility location")
    model.nofFacilities = len( installation )
    model.nofCustomers  = len( service )
    model.facilities = range( model.nofFacilities )
    model.customers  = range( model.nofCustomers )

    model.x = Var( model.facilities, within=Binary )
    model.y = Var( model.customers, model.facilities, within=Binary )

    model.obj = Objective ( expr = sum([installation[j]*model.x[j] for j in model.facilities])                          \
                                 + sum([service[i][j]*model.y[i,j] for i in model.customers for j in model.facilities]) \
                          , sense=minimize)

    def ServeIfOpen( model, j ):
        return sum([model.y[i,j] for i in model.customers]) <= model.nofCustomers*model.x[j]

    def ChooseOneFacility( model, i ):
        return sum([model.y[i,j] for j in model.facilities]) == 1

    model.serve  = Constraint( model.facilities, rule=ServeIfOpen )
    model.choose = Constraint( model.customers, rule=ChooseOneFacility )

    from pyomo.opt import SolverFactory
    results = solve_checked(model, solver)

    X = [ model.x[j].value >= .5 for j in model.facilities ]
    Y = [ [ model.y[i,j].value >= .5 for j in model.facilities ] for i in model.customers ]

    comparison_rows.append({'solver': solver, 'formulation': 'weak', 'objective': pyo.value(model.obj), 'seconds': perf_counter() - started})
    return X, Y, pyo.value(model.obj)

def FacilityLocationStrong( installation, service, solver=solver_name ):
    started = perf_counter()
    from pyomo.environ import ConcreteModel, Var, Objective, Constraint, NonNegativeReals, Binary, minimize

    model = ConcreteModel("Gina's facility location")
    model.nofFacilities = len( installation )
    model.nofCustomers  = len( service )
    model.facilities = range( model.nofFacilities )
    model.customers  = range( model.nofCustomers )

    model.x = Var( model.facilities, within=Binary )
    model.y = Var( model.customers, model.facilities, within=Binary )

    model.obj = Objective ( expr = sum([installation[j]*model.x[j] for j in model.facilities])                          \
                                 + sum([service[i][j]*model.y[i,j] for i in model.customers for j in model.facilities]) \
                          , sense=minimize)

    def ServeIfOpen( model, i, j ):
        return model.y[i,j] <= model.x[j]

    def ChooseOneFacility( model, i ):
        return sum([model.y[i,j] for j in model.facilities]) == 1

    model.serve  = Constraint( model.customers, model.facilities, rule=ServeIfOpen )
    model.choose = Constraint( model.customers, rule=ChooseOneFacility )

    from pyomo.opt import SolverFactory
    results = solve_checked(model, solver)

    X = [ model.x[j].value >= .5 for j in model.facilities ]
    Y = [ [ model.y[i,j].value >= .5 for j in model.facilities ] for i in model.customers ]

    comparison_rows.append({'solver': solver, 'formulation': 'strong', 'objective': pyo.value(model.obj), 'seconds': perf_counter() - started})
    return X, Y, pyo.value(model.obj)


In [ ]:
def GenerateFacilityLocationInstance( nofFacilities, nofCustumers, seed=42 ):
    facilities = range(nofFacilities)
    customers = range(nofCustumers)
    import numpy as np
    rng = np.random.default_rng(seed)
    xC = rng.integers( 0, 100, nofCustumers )
    yC = rng.integers( 0, 100, nofCustumers )
    xF = rng.integers( 0, 100, nofFacilities )
    yF = rng.integers( 0, 100, nofFacilities )

    installation = rng.integers( 100, 200, nofFacilities )

    dist = lambda i,j : ((xC[i]-xF[j])**2 + (yC[i]-yF[j])**2)

    service = [ [ dist(i,j) for j in facilities ] for i in customers ]

    return installation, service, xC, yC, xF, yF


In [ ]:
def ShowFacilityLocation( xC, yC, xF, yF, X=[], Y=[], cost=None ):
    import matplotlib.pyplot as plt
    [ plt.plot( [xC[i],xF[j]], [yC[i],yF[j]], 'g-' ) for j in range(len(X)) if X[j] for i in range(len(Y)) if Y[i][j] ]
    plt.plot( xC,yC, 'o' )
    plt.plot( xF,yF, 's' )
    if not cost is None:
        plt.title( str(cost) )
    plt.show()

In [ ]:
# The original large demonstration used (30, 500).
# This default is quick to repeat and fits all solver editions below.
installation, service, xC, yC, xF, yF = GenerateFacilityLocationInstance(8, 88)


In [ ]:
ShowFacilityLocation( xC, yC, xF, yF )

In [ ]:
ShowFacilityLocation(xC, yC, xF, yF, *FacilityLocationWeak(installation, service, solver='appsi_highs'))


In [ ]:
ShowFacilityLocation(xC, yC, xF, yF, *FacilityLocationStrong(installation, service, solver='appsi_highs'))


## Solver choice and formulation strength
Now run CBC on the same data. The original used GLPK and CBC; this portable edition compares HiGHS and CBC. A stronger LP relaxation can help a mixed-integer solver, but presolve, cuts and search strategies also affect runtime. These small timings do not establish a universal solver ranking.


In [ ]:
ShowFacilityLocation(xC, yC, xF, yF, *FacilityLocationWeak(installation, service, solver='cbc'))


In [ ]:
ShowFacilityLocation(xC, yC, xF, yF, *FacilityLocationStrong(installation, service, solver='cbc'))


## Install the commercial solver interfaces at this stage
The original lesson adds Gurobi, CPLEX and Xpress here, after the open-source comparison. We retain that sequence. Install only missing packages, then actually solve both formulations using each solver. The 8-facility, 88-customer example has 712 variables and at most 792 constraints.

The packaged small-model editions have documented limits: [Gurobi](https://support.gurobi.com/hc/en-us/articles/29682074018833-What-does-Restricted-license-for-non-production-use-only-mean), [CPLEX](https://www.ibm.com/products/ilog-cplex-optimization-studio/pricing), and [Xpress](https://github.com/fico-xpress/xpress-training/blob/main/python/md/python-full-course.md). If installation or licensing fails, resolve the error; a skipped solver is not a completed comparison.


In [ ]:
required_packages = {'gurobipy': 'gurobipy', 'cplex': 'cplex', 'xpress': 'xpress'}
ensure_packages(required_packages)
print(available_pyomo_solvers(['gurobi_direct', 'cplex_direct', 'xpress_direct']))


In [ ]:
for solver in ('gurobi_direct', 'cplex_direct', 'xpress_direct'):
    print(solver)
    ShowFacilityLocation(xC, yC, xF, yF, *FacilityLocationWeak(installation, service, solver=solver))
    ShowFacilityLocation(xC, yC, xF, yF, *FacilityLocationStrong(installation, service, solver=solver))
comparison = pd.DataFrame(comparison_rows)
assert len(comparison) == 10
assert comparison['solver'].nunique() == 5
assert comparison['objective'].max() - comparison['objective'].min() < 1e-4
display(comparison)
